In [13]:
from pathlib import Path
import re
from collections import Counter, defaultdict
from bs4 import BeautifulSoup
import sys


PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.config import SPLITS

In [8]:
# Parent labels
PARENT_LABELS = [
    "decision",
    "legislation",
    "secondary sources",
    "unable to classify"
]

# Child/sub labels
SUBLABELS = [
    "authors",
    "title",
    "fragment",
    "citation",
    "source",
]

In [11]:


per_split_totals = defaultdict(int)

for split in SPLITS:
    split_dir = PROJECT_ROOT /"data" / "annotated" / split

    if not split_dir.exists():
        print(f"skip (not found): {split_dir}")
        continue

    files = sorted(split_dir.rglob("*.html"))

    if not files:
        print(f"no html files in: {split_dir}")
        continue

    print(f"--- split: {split} (files: {len(files)}) ---")

    for f in files:
        try:
            text = f.read_text(encoding="utf-8", errors="ignore")

            soup = BeautifulSoup(text, "html.parser")

            # Find all manual_label and auto_label tags
            tags = soup.find_all(["manual_label", "auto_label"])

            count = len(tags)

            per_split_totals[split] += count

            rel = f.relative_to(PROJECT_ROOT)
            print(f"{rel}: {count}")

        except Exception as e:
            print(f"error processing {f}: {e}")

    print()

print("\n--- per-split totals ---")

for split in SPLITS:
    if split in per_split_totals:
        print(f"{split}: {per_split_totals[split]}")

--- split: train (files: 5) ---
data\annotated\train\1994CanLII4528NLCA.html: 424
data\annotated\train\1997CanLII16226ONCA.html: 1911
data\annotated\train\2008CSC9.html: 1399
data\annotated\train\2016NBOMB12.html: 306
data\annotated\train\2019SCC65.html: 4658

--- split: test (files: 3) ---
data\annotated\test\1989CanLII1415ONCA.html: 241
data\annotated\test\2005QCCA437.html: 312
data\annotated\test\2024NBKB203.html: 658

--- split: dev (files: 3) ---
data\annotated\dev\2001CanLII21117.html: 789
data\annotated\dev\2002SCC33.html: 953
data\annotated\dev\2021QCCA1675.html: 271

--- split: incoming (files: 10) ---
data\annotated\incoming\1993CanLII1889PESCTSD.html: 418
data\annotated\incoming\1993CanLII3004FCA.html: 1536
data\annotated\incoming\1999CanLII7320(ONCA).html: 1128
data\annotated\incoming\2001BCSC1342.html: 254
data\annotated\incoming\2003MBCA71.html: 134
data\annotated\incoming\2015NSSC25.html: 712
data\annotated\incoming\2020ABPLAB11.html: 203
data\annotated\incoming\2024BCSC

In [12]:


counts = Counter()

for split in SPLITS:
    split_dir = PROJECT_ROOT /"data" / "annotated" / split

    html_files = list(split_dir.rglob("*.html"))

    for html_file in html_files:
        try:
            with open(html_file, "r", encoding="utf-8") as f:
                soup = BeautifulSoup(f, "html.parser")

            # Find all manual_label and auto_label tags
            tags = soup.find_all(["manual_label", "auto_label"])

            for tag in tags:
                label = tag.get("labelname")

                if label:
                    counts[label] += 1

        except Exception as e:
            print(f"Error processing {html_file}: {e}")

# =========================
# PRINT FINAL TABLE
# =========================

print("\n" + "=" * 80)
print("TOTAL LABEL COUNTS")
print("=" * 80)

print(f"{'LABEL':<20} {'COUNT':>10}")
print("-" * 32)

# Parent labels
parent_total = 0
for label in PARENT_LABELS:
    c = counts[label]
    parent_total += c
    print(f"{label:<20} {c:>10}")

print("-" * 32)
print(f"{'TOTAL PARENT':<20} {parent_total:>10}")

print()

# Sublabels
sub_total = 0
for label in SUBLABELS:
    c = counts[label]
    sub_total += c
    print(f"{label:<20} {c:>10}")

print("-" * 32)
print(f"{'TOTAL SUBLABELS':<20} {sub_total:>10}")

print()

# Grand total
grand_total = parent_total + sub_total
print(f"{'TOTAL COUNT':<20} {grand_total:>10}")


TOTAL LABEL COUNTS
LABEL                     COUNT
--------------------------------
decision                   3235
legislation                2099
secondary sources           500
unable to classify           19
--------------------------------
TOTAL PARENT               5853

authors                     392
title                      4485
fragment                   3254
citation                   2815
source                      310
--------------------------------
TOTAL SUBLABELS           11256

TOTAL COUNT               17109


### # Word tokens per document

In [14]:
per_split_totals = defaultdict(int)
per_split_word_counts = defaultdict(list)  # Store word counts per file

for split in SPLITS:
    split_dir = PROJECT_ROOT / "data" / "annotated" / split

    if not split_dir.exists():
        print(f"skip (not found): {split_dir}")
        continue

    files = sorted(split_dir.rglob("*.html"))

    if not files:
        print(f"no html files in: {split_dir}")
        continue

    print(f"--- split: {split} (files: {len(files)}) ---")

    for f in files:
        try:
            text = f.read_text(encoding="utf-8", errors="ignore")
            soup = BeautifulSoup(text, "html.parser")

            # Extract raw text from the entire HTML
            raw_text = soup.get_text(separator=" ", strip=True)
            word_tokens = len(raw_text.split())

            # Store word count for this file
            rel = f.relative_to(PROJECT_ROOT)
            per_split_word_counts[split].append((rel, word_tokens))

            # Print file and word count
            print(f"{rel}: {word_tokens} word tokens")

        except Exception as e:
            print(f"error processing {f}: {e}")

    print()

# Print fancy summary
print("\n" + "="*60)
print("         WORD TOKEN COUNT SUMMARY         ")
print("="*60)
for split in SPLITS:
    if split in per_split_word_counts:
        print(f"\n--- {split.upper()} ---")
        for rel, count in per_split_word_counts[split]:
            print(f"  {rel}: {count}")
        total = sum(count for _, count in per_split_word_counts[split])
        print(f"  Total for {split}: {total} word tokens")
print("="*60)

--- split: train (files: 5) ---
data\annotated\train\1994CanLII4528NLCA.html: 10144 word tokens
data\annotated\train\1997CanLII16226ONCA.html: 40540 word tokens
data\annotated\train\2008CSC9.html: 28442 word tokens
data\annotated\train\2016NBOMB12.html: 7671 word tokens
data\annotated\train\2019SCC65.html: 65440 word tokens

--- split: test (files: 3) ---
data\annotated\test\1989CanLII1415ONCA.html: 4385 word tokens
data\annotated\test\2005QCCA437.html: 6570 word tokens
data\annotated\test\2024NBKB203.html: 16372 word tokens

--- split: dev (files: 3) ---
data\annotated\dev\2001CanLII21117.html: 17021 word tokens
data\annotated\dev\2002SCC33.html: 35285 word tokens
data\annotated\dev\2021QCCA1675.html: 4773 word tokens

--- split: incoming (files: 10) ---
data\annotated\incoming\1993CanLII1889PESCTSD.html: 10377 word tokens
data\annotated\incoming\1993CanLII3004FCA.html: 22255 word tokens
data\annotated\incoming\1999CanLII7320(ONCA).html: 23320 word tokens
data\annotated\incoming\2001B